# Upper Agent Scenario Topology And Load Matrix

Load a trained upper PPO agent, reset the 3-gNB scenario it was trained on, and plot the topology, target load matrix, current load matrix, agent bias matrix, and load matrix after one upper-control window.

Use the project virtual environment as the notebook kernel.

In [ ]:
from pathlib import Path
import json
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "global_ppo_3gnb_env.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from stable_baselines3 import PPO

from global_ppo_3gnb_env import DEFAULT_GNB_CONFIGS_3, GLOBAL_SNAPSHOT_SCENARIOS, GlobalPPO3GNBEnv, SLICE_TYPES

plt.rcParams.update({"figure.dpi": 125, "axes.grid": True, "grid.alpha": 0.22})
np.set_printoptions(precision=3, suppress=True)
print(f"Project root: {PROJECT_ROOT}")

## Select Run And Scenario

Leave `RUN_DIR = None` to use the newest upper-PPO run that has a saved model. Set `SCENARIO_NAME` to a snapshot name, or leave it as `None` to use the run config. If the config is `mixed`, the first curriculum snapshot is used by default.

In [ ]:
RUN_DIR = None
SCENARIO_NAME = None

def find_latest_run_with_model():
    run_root = PROJECT_ROOT / "models" / "upper_ppo_3gnb"
    candidates = []
    for run in run_root.glob("run_*"):
        if (run / "upper_ppo_final.zip").exists() or (run / "upper_ppo_best.zip").exists():
            candidates.append(run)
    if not candidates:
        raise FileNotFoundError("No saved upper PPO model found under models/upper_ppo_3gnb/run_*")
    return sorted(candidates, key=lambda p: p.stat().st_mtime)[-1]

run_dir = find_latest_run_with_model() if RUN_DIR is None else Path(RUN_DIR)
if not run_dir.is_absolute():
    run_dir = PROJECT_ROOT / run_dir

config_path = run_dir / "config.json"
config = json.loads(config_path.read_text()) if config_path.exists() else {}
model_path = run_dir / "upper_ppo_final.zip"
if not model_path.exists():
    model_path = run_dir / "upper_ppo_best.zip"

scenario_name = SCENARIO_NAME or config.get("snapshot_scenario", "mixed")
if scenario_name == "mixed":
    scenario_name = next(iter(GLOBAL_SNAPSHOT_SCENARIOS))

print(f"Run: {run_dir}")
print(f"Model: {model_path.name}")
print(f"Scenario: {scenario_name}")
print(json.dumps({k: config.get(k) for k in ['snapshot_scenario', 'snapshot_block_episodes', 'light_load_ues', 'medium_load_ues', 'high_load_ues', 'local_steps_per_global', 'global_steps_per_episode']}, indent=2))

## Create The Same Environment And Apply The Upper Agent

In [ ]:
env = GlobalPPO3GNBEnv(
    seed=int(config.get("seed", 7)),
    n_gnbs=int(config.get("n_gnbs", 3)),
    include_ue_counts=bool(config.get("include_ue_counts", True)),
    use_sumo_mobility=bool(config.get("use_sumo_mobility", False)),
    local_steps_per_global=int(config.get("local_steps_per_global", 10)),
    global_steps_per_episode=int(config.get("global_steps_per_episode", 12)),
    scenario_mode="snapshot",
    snapshot_scenario=scenario_name,
    terminal_reward_only=bool(config.get("terminal_reward_only", True)),
    use_progress_reward=bool(config.get("use_progress_reward", False)),
    max_handovers_per_local_step=int(config.get("max_handovers_per_local_step", 1)),
    action_direction_reward_weight=float(config.get("action_direction_reward_weight", 2.0)),
    snapshot_block_episodes=int(config.get("snapshot_block_episodes", 10)),
    light_load_ues=int(config.get("light_load_ues", 1)),
    medium_load_ues=int(config.get("medium_load_ues", 2)),
    high_load_ues=int(config.get("high_load_ues", 3)),
)

model = PPO.load(model_path, device="cpu")
obs, reset_info = env.reset()
action, _ = model.predict(obs, deterministic=True)
obs_after, reward, terminated, truncated, step_info = env.step(action)

target_matrix = np.asarray(reset_info["target_load_matrix"], dtype=float)
initial_load_matrix = np.asarray(reset_info["load_matrix"], dtype=float)
bias_matrix = np.asarray(step_info["bias_matrix"], dtype=float)
after_load_matrix = np.asarray(step_info["load_matrix"], dtype=float)
ue_count_matrix = np.asarray(step_info["ue_count_matrix"], dtype=float)

print(f"reward after one upper window: {reward:.3f}")
print(f"handovers in window: {step_info['handover_count']}")
print(f"target-load error: {step_info['target_load_error']:.3f}")

## Plot Helpers

In [ ]:
GNB_IDS = [0, 1, 2]
SLICE_COLORS = {"eMBB": "#2563eb", "URLLC": "#dc2626", "mMTC": "#16a34a"}
GNB_POS = {int(cfg["id"]): (float(cfg["x"]), float(cfg["y"]), float(cfg["coverage_radius"])) for cfg in DEFAULT_GNB_CONFIGS_3}

def draw_base_topology(ax):
    for gnb_id, (x, y, radius) in GNB_POS.items():
        ax.add_patch(Circle((x, y), radius, fill=False, linestyle="--", linewidth=1.2, alpha=0.35, color="#475569"))
        ax.scatter([x], [y], s=260, marker="^", color="#111827", zorder=5)
        ax.text(x, y + 28, f"gNB {gnb_id}", ha="center", va="bottom", fontweight="bold")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

def draw_topology_load(ax, matrix, title):
    offsets = {"eMBB": (-42, -30), "URLLC": (42, -30), "mMTC": (0, 48)}
    draw_base_topology(ax)
    for gnb_id, (x, y, _radius) in GNB_POS.items():
        for s_idx, slice_type in enumerate(SLICE_TYPES):
            value = float(matrix[gnb_id, s_idx])
            dx, dy = offsets[slice_type]
            ax.scatter([x + dx], [y + dy], s=260 + 1250 * value, color=SLICE_COLORS[slice_type], alpha=0.76, edgecolor="white", linewidth=1.1, zorder=6)
            ax.text(x + dx, y + dy, f"{value:.2f}", color="white", ha="center", va="center", fontsize=8, fontweight="bold", zorder=7)
    handles = [plt.Line2D([0], [0], marker="o", color="w", label=s, markerfacecolor=SLICE_COLORS[s], markersize=9) for s in SLICE_TYPES]
    ax.legend(handles=handles, loc="upper right")
    ax.set_title(title)

def heatmap(ax, matrix, title, cmap="viridis", vmin=0.0, vmax=1.0):
    im = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(SLICE_TYPES)), SLICE_TYPES)
    ax.set_yticks(range(len(GNB_IDS)), [f"gNB {g}" for g in GNB_IDS])
    ax.set_title(title)
    for r in range(matrix.shape[0]):
        for c in range(matrix.shape[1]):
            ax.text(c, r, f"{matrix[r, c]:.2f}", ha="center", va="center", color="white", fontweight="bold", fontsize=8)
    return im

## Topology: Target Loads The Upper Agent Trains On

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 6.6))
draw_topology_load(ax, target_matrix, f"Target load topology: {scenario_name}")
plt.show()

## Load Matrices Before And After The Upper Agent Action

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), constrained_layout=True)
heatmap(axes[0], target_matrix, "Target load", cmap="viridis", vmin=0, vmax=1)
heatmap(axes[1], initial_load_matrix, "Initial load", cmap="viridis", vmin=0, vmax=1)
heatmap(axes[2], bias_matrix, "Upper bias action", cmap="coolwarm", vmin=-1, vmax=1)
heatmap(axes[3], after_load_matrix, "After one window", cmap="viridis", vmin=0, vmax=1)
plt.show()

fig, ax = plt.subplots(figsize=(4.8, 4.0), constrained_layout=True)
heatmap(ax, ue_count_matrix, "UE count after one window", cmap="magma", vmin=0, vmax=max(1, ue_count_matrix.max()))
plt.show()

## UE Positions For The Scenario Reset

## Heuristic Verification: Upper Bias Direction

Loaded target cells should receive negative bias to encourage offload. Light target cells should receive positive bias to retain or attract load. Neutral cells are best near zero.

In [ ]:
rows = []
for gnb_id in GNB_IDS:
    for s_idx, slice_type in enumerate(SLICE_TYPES):
        target = float(target_matrix[gnb_id, s_idx])
        bias = float(bias_matrix[gnb_id, s_idx])
        if target >= 0.80:
            expected = "negative/offload"
            passed = bias < 0.0
        elif target <= 0.25:
            expected = "positive/retain"
            passed = bias > 0.0
        else:
            expected = "near zero/neutral"
            passed = abs(bias) <= 0.50
        rows.append({
            "gnb": gnb_id,
            "slice": slice_type,
            "target_load": target,
            "agent_bias": bias,
            "expected": expected,
            "pass": bool(passed),
        })

bias_check_df = pd.DataFrame(rows)
display(bias_check_df)
print(f"bias direction pass rate: {bias_check_df['pass'].mean():.2%}")

## Heuristic Verification: Lower A3 Offset Response

For the same radio/load state, the lower heuristic should be monotonic in the upper bias: `bias=-1` gives lower A3 offsets than `bias=0`, and `bias=+1` gives higher A3 offsets. Lower offsets make handover easier; higher offsets retain UEs.

In [ ]:
ue_counts = env._ue_count_dict()
slice_loads = env._slice_load_dict()
kmax = env._kmax_by_slice()

heuristic_rows = []
for gnb_id, agent in env.lower_agents.items():
    for slice_type in SLICE_TYPES:
        means = {}
        for bias_value in (-1.0, 0.0, 1.0):
            agent.reset()
            bias_row = {s: 0.0 for s in SLICE_TYPES}
            bias_row[slice_type] = bias_value
            offsets = agent.compute_offsets(
                bias_row=bias_row,
                ue_counts=ue_counts,
                kmax=kmax,
                slice_loads=slice_loads,
                handover_failure_ratios={},
                ping_pong_ratios={},
            )
            selected = [
                value["applied_offset_db"]
                for key, value in offsets.items()
                if key[0] == gnb_id and key[2] == slice_type
            ]
            means[bias_value] = float(np.mean(selected)) if selected else np.nan
        heuristic_rows.append({
            "gnb": gnb_id,
            "slice": slice_type,
            "serving_load": float(slice_loads[(gnb_id, slice_type)]),
            "offset_bias_-1": means[-1.0],
            "offset_bias_0": means[0.0],
            "offset_bias_+1": means[1.0],
            "monotonic_pass": bool(means[-1.0] <= means[0.0] <= means[1.0]),
        })

heuristic_check_df = pd.DataFrame(heuristic_rows)
display(heuristic_check_df)
print(f"lower heuristic monotonic pass rate: {heuristic_check_df['monotonic_pass'].mean():.2%}")

## Heuristic Verification: Actual Agent Offsets

This table shows the offsets produced by the lower heuristic for the actual upper-agent bias matrix. Negative offsets indicate easier handover away from the serving gNB/slice; positive offsets indicate stronger retention.

In [ ]:
actual_offset_rows = []
for gnb_id, agent in env.lower_agents.items():
    agent.reset()
    bias_row = {slice_type: float(bias_matrix[gnb_id, s_idx]) for s_idx, slice_type in enumerate(SLICE_TYPES)}
    offsets = agent.compute_offsets(
        bias_row=bias_row,
        ue_counts=ue_counts,
        kmax=kmax,
        slice_loads=slice_loads,
        handover_failure_ratios={},
        ping_pong_ratios={},
    )
    for (src, dst, slice_type), value in offsets.items():
        actual_offset_rows.append({
            "from_gnb": src,
            "to_gnb": dst,
            "slice": slice_type,
            "upper_bias": bias_row[slice_type],
            "serving_load": float(slice_loads[(src, slice_type)]),
            "target_load": float(slice_loads[(dst, slice_type)]),
            "proto_offset_db": float(value["proto_offset_db"]),
            "applied_offset_db": float(value["applied_offset_db"]),
        })

actual_offsets_df = pd.DataFrame(actual_offset_rows).sort_values(["from_gnb", "slice", "to_gnb"])
display(actual_offsets_df)

## Heuristic-Only Rollout Without The Upper Agent

This test does not call the PPO model. It uses a simple manual upper-bias rule: cells above the per-slice balance target get `-1` to encourage offload, cells below the target get `+1` to retain/accept load, and near-target cells get `0`. The lower heuristic then converts that bias matrix into A3 offsets.

In [ ]:
def make_heuristic_only_env():
    return GlobalPPO3GNBEnv(
        seed=int(config.get("seed", 7)),
        n_gnbs=int(config.get("n_gnbs", 3)),
        include_ue_counts=bool(config.get("include_ue_counts", True)),
        use_sumo_mobility=bool(config.get("use_sumo_mobility", False)),
        local_steps_per_global=int(config.get("local_steps_per_global", 10)),
        global_steps_per_episode=int(config.get("global_steps_per_episode", 12)),
        scenario_mode="snapshot",
        snapshot_scenario=scenario_name,
        terminal_reward_only=False,
        use_progress_reward=False,
        max_handovers_per_local_step=int(config.get("max_handovers_per_local_step", 1)),
        action_direction_reward_weight=0.0,
        snapshot_block_episodes=int(config.get("snapshot_block_episodes", 10)),
        light_load_ues=int(config.get("light_load_ues", 1)),
        medium_load_ues=int(config.get("medium_load_ues", 2)),
        high_load_ues=int(config.get("high_load_ues", 3)),
    )

def manual_balance_bias(load_matrix, balance_target_matrix, deadband=0.05):
    delta = np.asarray(load_matrix, dtype=float) - np.asarray(balance_target_matrix, dtype=float)
    bias = np.zeros_like(delta, dtype=float)
    bias[delta > deadband] = -1.0
    bias[delta < -deadband] = 1.0
    return bias

heur_env = make_heuristic_only_env()
heur_rows = []
heur_offset_rows = []

try:
    obs_h, info_h = heur_env.reset()
    first_load = np.asarray(info_h["load_matrix"], dtype=float)
    balance_target = np.asarray(info_h["balance_target_matrix"], dtype=float)
    target_for_plot = np.asarray(info_h["target_load_matrix"], dtype=float)

    for window in range(int(config.get("global_steps_per_episode", 12))):
        before_load = heur_env._load_matrix()
        before_error = heur_env._target_load_error(before_load)
        manual_bias = manual_balance_bias(before_load, balance_target)

        ue_counts_h = heur_env._ue_count_dict()
        slice_loads_h = heur_env._slice_load_dict()
        kmax_h = heur_env._kmax_by_slice()
        for gnb_id, agent in heur_env.lower_agents.items():
            agent.reset()
            bias_row = {s: float(manual_bias[gnb_id, s_idx]) for s_idx, s in enumerate(SLICE_TYPES)}
            offsets = agent.compute_offsets(
                bias_row=bias_row,
                ue_counts=ue_counts_h,
                kmax=kmax_h,
                slice_loads=slice_loads_h,
                handover_failure_ratios={},
                ping_pong_ratios={},
            )
            for (src, dst, slice_type), value in offsets.items():
                heur_offset_rows.append({
                    "window": window,
                    "from_gnb": src,
                    "to_gnb": dst,
                    "slice": slice_type,
                    "manual_bias": bias_row[slice_type],
                    "proto_offset_db": float(value["proto_offset_db"]),
                    "applied_offset_db": float(value["applied_offset_db"]),
                })

        obs_h, reward_h, terminated_h, truncated_h, info_h = heur_env.step(manual_bias.reshape(-1))
        after_load = np.asarray(info_h["load_matrix"], dtype=float)
        after_error = float(info_h["target_load_error"])
        heur_rows.append({
            "window": window,
            "reward": float(reward_h),
            "target_error_before": float(before_error),
            "target_error_after": after_error,
            "target_error_delta": float(before_error - after_error),
            "handover_count": int(info_h["handover_count"]),
            "sla_count": float(info_h["sla_count"]),
            "mean_abs_load_change": float(np.mean(np.abs(after_load - before_load))),
        })
        if terminated_h or truncated_h:
            break

    final_load = heur_env._load_matrix()
    final_ue_counts = np.asarray(info_h["ue_count_matrix"], dtype=float)
finally:
    heur_env.close()

heuristic_rollout_df = pd.DataFrame(heur_rows)
heuristic_offsets_df = pd.DataFrame(heur_offset_rows)
display(heuristic_rollout_df)
print(f"initial target error: {heuristic_rollout_df['target_error_before'].iloc[0]:.3f}")
print(f"final target error: {heuristic_rollout_df['target_error_after'].iloc[-1]:.3f}")
print(f"total handovers: {int(heuristic_rollout_df['handover_count'].sum())}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), constrained_layout=True)
heatmap(axes[0], target_for_plot, "Scenario target", cmap="viridis", vmin=0, vmax=1)
heatmap(axes[1], first_load, "Heuristic-only before", cmap="viridis", vmin=0, vmax=1)
heatmap(axes[2], final_load, "Heuristic-only after", cmap="viridis", vmin=0, vmax=1)
heatmap(axes[3], final_load - first_load, "Load change", cmap="coolwarm", vmin=-1, vmax=1)
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 4.0), constrained_layout=True)
ax.plot(heuristic_rollout_df["window"], heuristic_rollout_df["target_error_before"], marker="o", label="before window")
ax.plot(heuristic_rollout_df["window"], heuristic_rollout_df["target_error_after"], marker="o", label="after window")
ax.set_xlabel("upper window")
ax.set_ylabel("target-load error")
ax.set_title("Heuristic-only balance error")
ax.legend()
plt.show()

display(heuristic_offsets_df.head(36))

In [ ]:
ue_rows = []
for ue in env.base_env.get_all_ues():
    ue_rows.append({
        "ue_id": int(ue.id),
        "slice_type": str(getattr(ue, "slice_type", "eMBB")),
        "serving_gnb": int(ue.serving_gnb) if ue.serving_gnb is not None else -1,
        "x": float(ue.x),
        "y": float(ue.y),
    })
ue_df = pd.DataFrame(ue_rows).sort_values(["serving_gnb", "slice_type", "ue_id"])
display(ue_df)

fig, ax = plt.subplots(figsize=(8.4, 6.6))
draw_topology_load(ax, target_matrix, f"UE placement after one window: {scenario_name}")
for slice_type, rows in ue_df.groupby("slice_type"):
    ax.scatter(rows["x"], rows["y"], s=52, color=SLICE_COLORS.get(slice_type, "gray"), edgecolor="black", linewidth=0.5, zorder=9)
    for _, row in rows.iterrows():
        ax.text(row["x"] + 8, row["y"] + 8, f"u{int(row['ue_id'])}/g{int(row['serving_gnb'])}", fontsize=7)
plt.show()

In [ ]:
env.close()